# Semana 13 — Interpretabilidad desde la semana 12

Este notebook reemplaza el cuaderno provisional de dos celdas. Ejecutar de arriba abajo en **Colab CPU**.
Primero subir a la rama `semana-13` este notebook, `src/interpretabilidad.py` y `requirements_semana13.txt`.

Se reconstruye LightGBM con la configuración final congelada, muestreo del 50 % mensual,
semilla 42 y 1710 árboles. No se utilizan resultados del notebook 06 eliminado.
Los datos son los Parquet oficiales referenciados en `src/data.py`.

**Etapa 1:** entorno, datos, entrenamiento y guardado en Drive.
**Etapa 2:** control de reproducción en junio. Si falla, detenerse y revisar la causa.
**Etapa 3:** importancia, SHAP, errores por segmentos e informe con resultados de esta ejecución.

El preprocesamiento usa bloques y una matriz float64 en disco. Esto reduce el consumo de RAM;
no garantiza un tiempo de ejecución o memoria máxima. El entrenamiento puede ser prolongado en CPU.
No cambies el muestreo, los hiperparámetros ni las tolerancias a partir de los resultados de junio.

## 1. Obtener la rama de trabajo
Se usa una carpeta nueva para no modificar `/content/proy-machine`, creada en los pasos anteriores.
Esta celda lee GitHub; no hace commits ni push. Si ya existe la carpeta, conserva su versión local.

In [ ]:
from pathlib import Path
import subprocess
import os
import sys

REPO_DIR = Path('/content/proy-machine-s13')
REPO_URL = 'https://github.com/A-Joacox/proy-machine.git'
RAMA = 'semana-13'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', RAMA, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / '.git').is_dir():
    raise RuntimeError('La carpeta de trabajo no es un repositorio Git.')
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
for args in [['git', 'log', '-1', '--oneline'], ['git', 'branch', '--show-current']]:
    print(subprocess.check_output(args, text=True).strip())
for name in ['src/interpretabilidad.py', 'requirements_semana13.txt']:
    if not (REPO_DIR / name).exists():
        raise FileNotFoundError(f'Primero sube {name} a la rama semana-13 de GitHub.')

## 2. Instalar el entorno comprobado
Estas versiones se usan para esta reconstrucción; no se afirma que sean las versiones originales de semana 12.
Si Colab pide reiniciar la sesión tras instalar, hazlo y vuelve a ejecutar desde la sección 1.
La celda siguiente detecta librerías antiguas que hayan quedado cargadas en memoria.

In [ ]:
%pip install -q -r /content/proy-machine-s13/requirements_semana13.txt

In [ ]:
import importlib
import importlib.metadata

for package, module in [('numpy','numpy'), ('pandas','pandas'), ('scipy','scipy'),
                        ('scikit-learn','sklearn'), ('lightgbm','lightgbm'),
                        ('pyarrow','pyarrow'), ('shap','shap'), ('joblib','joblib'),
                        ('matplotlib','matplotlib')]:
    loaded = importlib.import_module(module)
    installed = importlib.metadata.version(package)
    if loaded.__version__ != installed:
        raise RuntimeError(f'{package}: reinicia la sesión de Colab y vuelve a ejecutar desde la sección 1.')
    print(f'{package}: {installed}')

## 3. Confirmar la base de semana 12 y los recursos
Se verifican los hashes de los cinco archivos de partida leídos del ZIP. Una diferencia detiene la
reproducción para no mezclar configuraciones. Los resultados viejos de semana 13 no se consultan.

In [ ]:
import json
import hashlib
import psutil
import shutil
import numpy as np
import pandas as pd

EXPECTED_HASHES = {'src/data.py': '4259180c087d865ebd2c803e21ad0f5add97b8bea434bd19b56a1152b7d64ab4', 'src/features.py': 'e1045f83b0d2a5fb0fbc4ea95879761d4d9bd7e179d9e3a8bd6b92ab526a85fd', 'outputs/config_modelo_final.json': '9ed5e35043515ae318c7aa6bce36126bfca206d67119f8e3b1436526822510a3', 'outputs/metrics_evaluacion_final.json': 'ac95cad610eefe73b34d61af5d14389bdd96cf231aeba612aaf5975dcba78ded', 'data/Taxi_Zone_Lookup.csv': '1a99e105092230f8620f301edcca7f80d3080642ff404d28ed957d3fa222c8ed'}
for relative, expected in EXPECTED_HASHES.items():
    actual = hashlib.sha256((REPO_DIR / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Cambió {relative} respecto a la semana 12 revisada. Detente y comparte este mensaje.')

config = json.loads((REPO_DIR / 'outputs/config_modelo_final.json').read_text())
reference = json.loads((REPO_DIR / 'outputs/metrics_evaluacion_final.json').read_text())['resumen']
assert config['sample_frac'] == 0.5 and config['seed'] == 42 and config['n_estimators'] == 1710
print(json.dumps(config, indent=2, ensure_ascii=False))
print(f"Train esperado: {reference['n_train']:,}; test esperado: {reference['n_test']:,}")
print(f"RAM disponible: {psutil.virtual_memory().available / 1024**3:.1f} GiB")
print(f"Disco libre: {shutil.disk_usage('/content').free / 1024**3:.1f} GiB")

## 4. Guardar el modelo y resultados en Drive
Autoriza tu propia cuenta cuando Colab lo solicite. Se crea una carpeta específica para esta ejecución nueva.
El archivo joblib se carga únicamente de esta carpeta creada por ti.
Si el checkpoint ya existe y coincide en código y versiones, se reutiliza sin entrenar.
Si no coincide o está incompleto, el código se detiene y lo conserva.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/ML_NYC_semana13_desde_semana12')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DRIVE_DIR / 'modelo'
WORK_DIR = Path('/content/semana13_temporal')
WORK_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = REPO_DIR / 'outputs/semana13'
REPORT_DIR = REPO_DIR / 'reports/semana13'
FIGURES_DIR = REPORT_DIR / 'figures'
for path in [OUTPUT_DIR, REPORT_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', CHECKPOINT_DIR)
print('Resultados nuevos:', OUTPUT_DIR)

## 5. Reconstruir el modelo y guardarlo
Esta es la celda larga. Descarga enero-mayo, aplica exactamente las reglas y el muestreo originales,
verifica equivalencia de preprocesamiento en filas reales y entrena 1710 árboles.
No carga junio durante el ajuste. Muestra progreso cada 100 árboles.

El modelo se guarda al terminar el entrenamiento. Si la sesión se interrumpe antes de ese guardado,
no hay todavía un modelo completo que recuperar. Una vez guardado, esta misma celda lo carga de Drive.

In [ ]:
from src.interpretabilidad import (
    reconstruir_modelo, evaluar_junio, analisis_segmentos,
    interpretar, escribir_informe, guardar_json,
)

modelo, preprocesador, metadata_modelo = reconstruir_modelo(
    REPO_DIR, CHECKPOINT_DIR, WORK_DIR, config,
    expected_n_train=reference['n_train'],
)
guardar_json(OUTPUT_DIR / 'modelo_metadata.json', metadata_modelo)
print('Listo: modelo y preprocesador disponibles y guardados en Drive.')

## 6. Verificar la reproducción sobre junio
El conjunto de junio conserva el 50 % y la semilla 42, con el número de viajes esperado de semana 12.
La inferencia se hace por bloques para no mantener toda la matriz transformada en memoria.

Controles fijados antes de ejecutar: diferencia absoluta MAE ≤ 0.02 minutos y RMSLE ≤ 0.002.
Estos límites son controles operativos, no intervalos estadísticos ni prueba de identidad de modelos.
Si hay una discrepancia, guarda el mensaje y revisa datos, librerías y código. No cambies parámetros
para mejorar el test. Las métricas de semana 12 permanecen en su archivo original.

In [ ]:
test, verificacion = evaluar_junio(
    modelo, preprocesador, config, reference, OUTPUT_DIR,
)
shutil.copytree(OUTPUT_DIR, DRIVE_DIR / 'resultados/outputs/semana13', dirs_exist_ok=True)
print('Control completado. Puedes compartir esta salida antes de continuar con SHAP.')

## 7. Errores y cobertura por segmento
Se usa todo junio del test. Se reportan tamaño del grupo, MAE, error medio firmado y proporción de
subestimaciones. El error firmado es predicción − real. Un MAE alto no prueba subestimación.
Los tramos de distancia real son un análisis posterior; no se usa esa distancia como predictor.

In [ ]:
segmentos = analisis_segmentos(test, preprocesador, OUTPUT_DIR)
display(segmentos)

## 8. Importancia y SHAP
Muestra aleatoria simple **predefinida de 2000 viajes**, semilla 42, del test de junio.
La muestra solo se utiliza para explicar; no cambia el entrenamiento ni el test completo de métricas.

TreeSHAP con `tree_path_dependent` y salida `raw` explica **log(1 + duración en minutos)**.
Se comprueba la suma de contribuciones. No se convierte cada SHAP a minutos por separado.
Las importancias no son efectos causales; columnas correlacionadas pueden repartirse la atribución.
La selección de ejemplos de bajo, mediano y alto error es deliberada y no representa frecuencias poblacionales.

In [ ]:
N_SHAP = 2000
SEED_SHAP = 42
importancias, metadata_shap = interpretar(
    modelo, preprocesador, test, OUTPUT_DIR, FIGURES_DIR,
    n_shap=N_SHAP, seed=SEED_SHAP,
)
display(importancias.sort_values('mean_abs_shap_log1p', ascending=False).head(15))
print('Máximo error de aditividad:', metadata_shap['error_max_aditividad'])

## 9. Figuras, informe y copia de respaldo
El informe se genera con cifras de esta ejecución y limitaciones explícitas. No contiene conclusiones
numéricas copiadas del notebook eliminado. Revisa sus tablas y figuras antes de hacer el commit final.

In [ ]:
from IPython.display import display, Image, Markdown

informe = escribir_informe(
    REPORT_DIR / 'informe_semana13.md', verificacion, importancias,
    metadata_shap, segmentos, metadata_modelo,
)
shutil.copytree(OUTPUT_DIR, DRIVE_DIR / 'resultados/outputs/semana13', dirs_exist_ok=True)
shutil.copytree(REPORT_DIR, DRIVE_DIR / 'resultados/reports/semana13', dirs_exist_ok=True)
for path in sorted(FIGURES_DIR.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))
display(Markdown(informe.read_text(encoding='utf-8')))

## 10. Descargar resultados para GitHub
Este ZIP contiene únicamente los resultados e informe nuevos, con sus rutas. No incluye los Parquet
crudos ni el modelo. Extrae el ZIP en tu computadora y sube sus archivos a la rama `semana-13`.
Además, guarda/descarga este notebook con sus salidas y actualiza `notebooks/06_interpretabilidad.ipynb`.

Commit del código: `Semana 13: implementar reconstrucción e interpretabilidad desde semana 12`.
Commit después de revisar la ejecución: `Semana 13: añadir resultados SHAP, sesgos y limitaciones`.
No marques la semana como ejecutada antes de completar las secciones 5–9.

In [ ]:
import zipfile
from google.colab import files

archive = Path('/content/semana13_resultados.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in [OUTPUT_DIR, REPORT_DIR]:
        for path in sorted(folder.rglob('*')):
            if path.is_file():
                z.write(path, path.relative_to(REPO_DIR))
files.download(str(archive))